# TP3 · Preparación, modelado y datos para el dashboard

**Versión unificada · 17/11/2026.** Integra las dos versiones adjuntas y reemplaza sus resultados escritos a mano por tablas calculadas. Es autónoma: **no requiere `tp3_config.py`**. Incluye la comparación de modelos presente en los originales para producir los datos que consumirá el dashboard; no construye todavía su interfaz.

## Cómo ejecutarlo
1. Guardar este archivo en `TP3/notebooks/` (o en `TP3/`).
2. Colocar los 61 CSV `forecast_observation_YYYY_MM_DD.csv` en `TP3/dataset/`.
3. Usar un entorno con Python 3.10+, pandas 2+, NumPy, scikit-learn 1.2+, matplotlib y joblib. En una terminal: `python -m pip install pandas numpy scikit-learn matplotlib joblib jupyter`.
4. Reiniciar el kernel y ejecutar todas las celdas, en orden. No continuar si falla un control.
5. Consumir **una única ejecución completa** de `TP3/resultados_dashboard_04/<id_ejecucion>/`. El archivo `manifest.json` se escribe al final: su ausencia indica una ejecución incompleta.

No se incluyen resultados nuevos sobre los CSV reales: no fueron adjuntados. Los valores 854.009 y 140.822 de versiones anteriores son referencias, no condiciones de validez ni resultados de esta versión. Los notebooks 05 y 06 no fueron verificados en esta entrega.

## Preguntas para la presentación
- ¿Cuánto error tiene el pronóstico original y cuánto cambia con la corrección?
- ¿La mejora se sostiene por día, horizonte y estación?
- ¿Dónde empeora y cuántas observaciones respaldan cada resultado?
- ¿Cuándo faltan antecedentes y cómo afecta eso a la corrección?

Error con signo = observado − pronosticado. Un sesgo positivo indica subestimación de temperatura. Corrección = pronóstico + error estimado.

**Verificación de esta entrega:** sintaxis de todas las celdas y ejecución completa con 61 CSV sintéticos (incluidos entrenamiento, evaluación y exportación). Pruebas específicas de disponibilidad estricta, aislamiento por estación, vencimiento de lags, invariancia ante etiquetas futuras y reagrupación de métricas. También se probó el caso en que gana el pronóstico original. Esto comprueba funcionamiento del código con datos de prueba; no reproduce las métricas de los CSV reales.


## 1. Protocolo fijo antes de observar resultados

| Uso | Fechas objetivo UTC |
|---|---|
| Entrenamiento candidato | 10/09–17/10/2026 |
| Validación para seleccionar una configuración | 18/10–01/11/2026 |
| Exclusión por anomalías previamente identificadas | 22–26/10/2026 |
| Test retrospectivo reservado | 02–09/11/2026 |

Los 61 archivos se distribuyen en 48 históricos, 5 excluidos y 8 de test. Se consideran horizontes nominales de 12 y 24 h, emparejamiento `interval` y temperaturas finitas.

**Purga:** una etiqueta de entrenamiento debe estar disponible estrictamente antes de la primera emisión de validación. Para el ajuste final se aplica la misma regla antes del 01/11 a las 00:00 UTC y se comprueba que ninguna emisión de test sea anterior a ese corte.

**Lags secuenciales:** emisión efectiva = máximo entre generación y recolección del pronóstico; disponibilidad del error = máximo entre emisión efectiva y recolección de la observación + 2 h. Se busca el último evento disponible de la misma estación y horizonte, estrictamente anterior a la emisión, con objetivo anterior y edad máxima de 72 h. Si el evento más recientemente disponible es demasiado antiguo, el lag queda ausente; no se busca otro sustituto. Los empates se resuelven por objetivo más reciente.

La purga limita el ajuste de parámetros, no el historial operativo. Un error de una fila purgada o de una fecha de validación/test puede alimentar una predicción posterior **solo después de su disponibilidad estimada**. Esta misma regla se usa en histórico y test. En 06, cada fold debe purgar el ajuste y mantener esta política de eventos: restringir los lags a las filas de entrenamiento sería otro experimento. B3 u otros baselines aprendidos deben ajustarse dentro de cada fold.

**Alcance:** la selección usa validación hasta el 01/11 y algunas emisiones de test pertenecen a ese día. Además, versiones anteriores ya exploraron este test. Se informa como evaluación retrospectiva reservada por fecha objetivo, no como prueba prospectiva inédita. Los parámetros quedan congelados durante test; sus observaciones solo actualizan antecedentes permitidos. El margen de 2 h es una hipótesis, no una medición exacta de publicación.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

SEED = 42
MARGEN_H = 2
EDAD_MAX_H = 72
CORTE_VAL = pd.Timestamp('2026-10-18', tz='UTC')
INICIO_TEST = pd.Timestamp('2026-11-02', tz='UTC')
CORTE_FINAL = pd.Timestamp('2026-11-01', tz='UTC')
VERSION = '04_dashboard_v1'

def exigir(condicion, mensaje):
    if not bool(condicion):
        raise ValueError(mensaje)

RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'dataset').is_dir()), None)
exigir(RAIZ is not None, 'No se encontró TP3/dataset/. Abrir desde TP3/ o TP3/notebooks/.')
DATASET = RAIZ / 'dataset'
esperadas = pd.date_range('2026-09-10', '2026-11-09', freq='D')
excluidas = set(pd.date_range('2026-10-22', '2026-10-26', freq='D'))
nombres = {f'forecast_observation_{d:%Y_%m_%d}.csv': d for d in esperadas}
encontrados = {p.name: p for p in DATASET.glob('forecast_observation_*.csv')}
exigir(set(encontrados) == set(nombres),
       f'Inventario incorrecto. Faltantes: {sorted(set(nombres)-set(encontrados))}; '
       f'inesperados: {sorted(set(encontrados)-set(nombres))}')
inventario = pd.DataFrame([
    {'archivo': nombre, 'fecha': str(d.date()),
     'uso': 'excluido' if d in excluidas else ('test' if d >= pd.Timestamp('2026-11-02') else 'historico'),
     'bytes': encontrados[nombre].stat().st_size,
     'mtime_ns': encontrados[nombre].stat().st_mtime_ns}
    for nombre, d in nombres.items()
])
id_ejecucion = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
SALIDA = RAIZ / 'resultados_dashboard_04' / id_ejecucion
SALIDA.mkdir(parents=True, exist_ok=False)
display(inventario.groupby('uso').size().rename('archivos').to_frame())
print('Salidas de esta ejecución:', SALIDA)


## 2. Lectura única y controles de calidad

Cada CSV incluido se lee una vez, por bloques. Se conserva `collection_time_utc` desde el inicio y se calculan disponibilidad y calidad a partir de los datos cargados. Los cinco archivos excluidos se inventarían pero no se usan para ajustar, evaluar ni generar antecedentes.

Primero se filtran los registros elegibles. Sobre ellos se exige consistencia de fechas, horizontes, intervalos, recolección y claves. Un fallo detiene el proceso: no se eliminan silenciosamente filas problemáticas. La falta de predictores meteorológicos se imputa dentro de cada entrenamiento; latitud/longitud y marcas temporales son obligatorias. Los conteos de descarte corresponden al filtro de elegibilidad, no a un juicio de calidad sobre todos los registros originales.


In [ ]:
FECHAS = ['snapshot_time_utc', 'forecast_generated_at_utc', 'collection_time_utc',
          'target_time_utc', 'observation_time_utc', 'observation_collection_hour_utc',
          'forecast_start_utc', 'forecast_end_utc']
NUMERICAS = ['latitude', 'longitude', 'horizon_target_hours', 'temperature_forecast_c',
             'temperature_observed_c', 'relative_humidity_forecast_pct', 'dewpoint_forecast_c',
             'wind_speed_forecast_kmh', 'precip_probability_forecast_pct']
COLUMNAS = ['station_id', 'state', 'match_type', 'wind_direction_forecast'] + FECHAS + NUMERICAS
CLAVE = ['station_id', 'horizon_target_hours', 'target_time_utc']

def cargar(uso):
    partes, auditoria = [], []
    for fila in inventario.loc[inventario.uso.eq(uso)].itertuples():
        total = retenidas = 0
        for b in pd.read_csv(DATASET / fila.archivo, usecols=COLUMNAS,
                             dtype={'station_id': 'string', 'state': 'string'}, chunksize=100_000):
            total += len(b)
            for c in NUMERICAS:
                b[c] = pd.to_numeric(b[c], errors='coerce')
            elegible = (b.horizon_target_hours.isin([12, 24]) & b.match_type.eq('interval')
                        & np.isfinite(b.temperature_forecast_c) & np.isfinite(b.temperature_observed_c))
            b = b.loc[elegible].copy()
            if b.empty:
                continue
            for c in FECHAS:
                b[c] = pd.to_datetime(b[c], utc=True, errors='coerce')
            exigir(b[FECHAS].notna().all().all(), f'{fila.archivo}: fechas faltantes/inválidas')
            exigir((b.station_id.notna() & b.station_id.str.strip().ne('')).all(), 'Estación faltante')
            exigir(b.target_time_utc.dt.strftime('%Y-%m-%d').eq(fila.fecha).all(), 'Fecha objetivo ≠ archivo')
            h = (b.target_time_utc-b.snapshot_time_utc).dt.total_seconds()/3600
            exigir(np.isclose(h, b.horizon_target_hours, atol=.001, rtol=0).all(), 'Horizonte inconsistente')
            exigir((b.observation_time_utc.ge(b.forecast_start_utc)
                    & b.observation_time_utc.lt(b.forecast_end_utc)).all(), 'Observación fuera del intervalo')
            exigir(b.observation_collection_hour_utc.ge(b.observation_time_utc).all(), 'Recolección anterior a observación')
            exigir((b.latitude.between(-90, 90) & b.longitude.between(-180, 180)).all(), 'Coordenadas inválidas')
            b['momento_emision_utc'] = b[['collection_time_utc', 'forecast_generated_at_utc']].max(axis=1)
            exigir(b.momento_emision_utc.lt(b.target_time_utc).all(), 'Emisión no anterior al objetivo')
            b['error_disponible_utc'] = pd.concat([
                b.momento_emision_utc,
                b.observation_collection_hour_utc + pd.Timedelta(hours=MARGEN_H)
            ], axis=1).max(axis=1)
            b['error_c'] = b.temperature_observed_c - b.temperature_forecast_c
            b['archivo_origen'] = fila.archivo
            b['origen_evento'] = uso
            b['fecha'] = b.target_time_utc.dt.strftime('%Y-%m-%d')
            b['horizon_target_hours'] = b.horizon_target_hours.astype('int64')
            # Infinitos en predictores son faltantes; se reportan en la disponibilidad.
            b[NUMERICAS] = b[NUMERICAS].replace([np.inf, -np.inf], np.nan)
            retenidas += len(b)
            partes.append(b)
        auditoria.append({'archivo': fila.archivo, 'fecha': fila.fecha, 'uso': uso,
                          'filas_leidas': total, 'filas_elegibles': retenidas,
                          'filas_no_elegibles': total-retenidas})
        exigir(retenidas > 0, f'{fila.archivo}: sin filas elegibles')
    exigir(bool(partes), f'Sin datos para {uso}')
    df = pd.concat(partes, ignore_index=True)
    exigir(not df.duplicated(CLAVE).any(), f'{uso}: claves estación/horizonte/objetivo duplicadas')
    cobertura = pd.crosstab(df.fecha, df.horizon_target_hours).reindex(columns=[12, 24], fill_value=0)
    exigir((cobertura > 0).all().all(), f'{uso}: falta algún horizonte por día')
    return df, pd.DataFrame(auditoria)

historico, calidad_historico = cargar('historico')
disponibilidad = historico[NUMERICAS + ['wind_direction_forecast']].isna().sum().rename('faltantes').to_frame()
disponibilidad['registros'] = len(historico)
disponibilidad['disponibilidad_pct'] = 100 * (1-disponibilidad.faltantes/len(historico))
disponibilidad.index.name = 'variable'
display(calidad_historico)
display(disponibilidad)


## 3. Transformaciones compartidas y lags auditables

La misma función transforma histórico y test, sin ajustar estadísticas. La hora solar es una aproximación por longitud, no la hora civil ni una efeméride. El viento en calma tiene componentes cero; una dirección no reconocida con viento positivo detiene la ejecución. Las columnas de trazabilidad del lag se conservan para auditoría y no se usan como predictores.


In [ ]:
BASE = ['temperature_forecast_c', 'dewpoint_forecast_c', 'relative_humidity_forecast_pct',
        'wind_speed_forecast_kmh', 'precip_probability_forecast_pct', 'latitude', 'longitude',
        'horizon_target_hours', 'depresion_rocio_c', 'inversion_rocio_flag', 'humedad_cero_flag',
        'viento_calma_flag', 'direccion_viento_sin', 'direccion_viento_cos',
        'hora_utc_sin', 'hora_utc_cos', 'hora_solar_sin', 'hora_solar_cos', 'dia_anio_sin', 'dia_anio_cos']
LAGS = ['lag_error_disponible_c', 'lag_error_edad_h', 'lag_error_faltante_flag']
PREDICTORES = BASE + LAGS

def construir_features_meteorologicas(df):
    r = df.copy()
    r['depresion_rocio_c'] = (r.temperature_forecast_c-r.dewpoint_forecast_c).clip(lower=0)
    r['inversion_rocio_flag'] = r.dewpoint_forecast_c.gt(r.temperature_forecast_c).astype('int8')
    r['humedad_cero_flag'] = r.relative_humidity_forecast_pct.eq(0).astype('int8')
    r['viento_calma_flag'] = r.wind_speed_forecast_kmh.eq(0).astype('int8')
    mapa = dict(zip('N NNE NE ENE E ESE SE SSE S SSW SW WSW W WNW NW NNW'.split(), np.arange(16)*22.5))
    grados = r.wind_direction_forecast.astype('string').str.strip().str.upper().map(mapa).astype(float)
    exigir(~(r.wind_speed_forecast_kmh.gt(0) & grados.isna()).any(), 'Dirección desconocida con viento positivo')
    angulo = np.deg2rad(grados)
    r['direccion_viento_sin'] = np.sin(angulo).fillna(0).mask(r.viento_calma_flag.eq(1), 0)
    r['direccion_viento_cos'] = np.cos(angulo).fillna(0).mask(r.viento_calma_flag.eq(1), 0)
    hora = r.target_time_utc.dt.hour + r.target_time_utc.dt.minute/60
    solar = (hora + r.longitude/15) % 24
    dias_anio = np.where(r.target_time_utc.dt.is_leap_year, 366, 365)
    for prefijo, fase in [('hora_utc', hora/24), ('hora_solar', solar/24),
                          ('dia_anio', (r.target_time_utc.dt.dayofyear-1)/dias_anio)]:
        r[prefijo+'_sin'] = np.sin(2*np.pi*fase)
        r[prefijo+'_cos'] = np.cos(2*np.pi*fase)
    exigir(not np.isinf(r[BASE].to_numpy(float)).any(), 'Infinitos en features')
    return r

def incorporar_lags(consultas, fuente):
    grupos = ['station_id', 'horizon_target_hours']
    r = consultas.reset_index(drop=True).copy()
    q = r[grupos + ['momento_emision_utc']].copy()
    q['orden'] = np.arange(len(q))
    e = fuente[grupos + ['error_disponible_utc', 'target_time_utc', 'error_c', 'origen_evento']].rename(columns={
        'error_disponible_utc': 'lag_disponible_utc', 'target_time_utc': 'lag_objetivo_utc',
        'error_c': 'lag_error_disponible_c', 'origen_evento': 'lag_origen'})
    e = e.sort_values(['lag_disponible_utc', 'lag_objetivo_utc'], kind='stable')
    lag = pd.merge_asof(q.sort_values('momento_emision_utc'), e,
                        by=grupos, left_on='momento_emision_utc', right_on='lag_disponible_utc',
                        direction='backward', allow_exact_matches=False).sort_values('orden').reset_index(drop=True)
    edad = (lag.momento_emision_utc-lag.lag_objetivo_utc).dt.total_seconds()/3600
    valido = (lag.lag_error_disponible_c.notna() & lag.lag_objetivo_utc.lt(lag.momento_emision_utc)
              & edad.between(0, EDAD_MAX_H))
    exigir(lag.loc[valido, 'lag_disponible_utc'].lt(lag.loc[valido, 'momento_emision_utc']).all(), 'Fuga temporal en lag')
    for col in ['lag_error_disponible_c', 'lag_disponible_utc', 'lag_objetivo_utc', 'lag_origen']:
        r[col] = lag[col].where(valido)
    r['lag_error_edad_h'] = edad.where(valido)
    r['lag_error_faltante_flag'] = (~valido).astype('int8')
    return r

features_historicas = incorporar_lags(construir_features_meteorologicas(historico), historico)
exigir(not set(PREDICTORES) & {'error_c', 'temperature_observed_c', 'observation_time_utc'}, 'Objetivo entre predictores')
display(features_historicas.groupby('horizon_target_hours').agg(
    registros=('station_id', 'size'), sin_lag=('lag_error_faltante_flag', 'sum')))


## 4. Separación temporal y comparación de configuraciones

Se comparan el pronóstico original, Ridge y HGB, con y sin lags. Son configuraciones fijas heredadas de los originales, no una búsqueda exhaustiva. Se selecciona por el promedio de MAE de los dos horizontes en validación, para darles el mismo peso. El original también puede ganar: no se fuerza una mejora.

Medianas y escalas se ajustan solo con entrenamiento purgado. Los lags ausentes reciben cero y conservan su indicador. HGB no usa validación interna aleatoria (`early_stopping=False`). Se mantienen los 23 predictores; el cero explícito en viento en calma puede cambiar métricas respecto a versiones anteriores.


In [ ]:
val = features_historicas.loc[features_historicas.target_time_utc.ge(CORTE_VAL)].copy()
train_candidato = features_historicas.loc[features_historicas.target_time_utc.lt(CORTE_VAL)].copy()
exigir(not val.empty, 'Validación vacía')
inicio_val = val.momento_emision_utc.min()
train = train_candidato.loc[train_candidato.error_disponible_utc.lt(inicio_val)].copy()
exigir(not train.empty, 'Entrenamiento vacío después de purga')
exigir(set(train.horizon_target_hours) == {12, 24}, 'Falta horizonte en entrenamiento')
resumen_particiones = pd.DataFrame([
    {'conjunto': 'train_candidato', 'filas': len(train_candidato)},
    {'conjunto': 'train_purgado', 'filas': len(train)},
    {'conjunto': 'purgadas', 'filas': len(train_candidato)-len(train)},
    {'conjunto': 'validacion', 'filas': len(val)}])
display(resumen_particiones)
print('Etiquetas de entrenamiento estrictamente anteriores a:', inicio_val)

def crear_pipeline(tipo, usar_lags):
    base = Pipeline([('imputar', SimpleImputer(strategy='median', keep_empty_features=True)),
                     ('escalar', StandardScaler() if tipo == 'Ridge' else 'passthrough')])
    transformaciones = [('base', base, BASE)]
    if usar_lags:
        lag = Pipeline([('imputar', SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True)),
                        ('escalar', StandardScaler() if tipo == 'Ridge' else 'passthrough')])
        transformaciones.append(('lags', lag, LAGS))
    modelo = (Ridge(alpha=10., solver='lsqr', max_iter=2000, tol=1e-5) if tipo == 'Ridge' else
              HistGradientBoostingRegressor(learning_rate=.07, max_iter=120, max_leaf_nodes=15,
                  max_depth=6, min_samples_leaf=80, l2_regularization=10., early_stopping=False, random_state=SEED))
    return Pipeline([('preprocesador', ColumnTransformer(transformaciones)), ('modelo', modelo)])

CONFIGURACIONES = {'Ridge_base': ('Ridge', False), 'Ridge_lags': ('Ridge', True),
                   'HGB_base': ('HGB', False), 'HGB_lags': ('HGB', True)}
modelos = {}
pred_val = {'Original': np.zeros(len(val))}
for nombre, (tipo, usa_lags) in CONFIGURACIONES.items():
    columnas = PREDICTORES if usa_lags else BASE
    modelo = crear_pipeline(tipo, usa_lags)
    modelo.fit(train[columnas], train.error_c)
    pred_val[nombre] = modelo.predict(val[columnas])
    exigir(np.isfinite(pred_val[nombre]).all(), f'{nombre}: predicciones no finitas')
    modelos[nombre] = modelo
    print('Entrenado:', nombre)

def detalle_predicciones(df, predicciones, conjunto):
    columnas = ['station_id', 'state', 'latitude', 'longitude', 'fecha', 'target_time_utc',
                'momento_emision_utc', 'horizon_target_hours', 'temperature_forecast_c',
                'temperature_observed_c', 'lag_error_faltante_flag', 'lag_error_edad_h',
                'lag_disponible_utc', 'lag_objetivo_utc', 'lag_origen', 'archivo_origen']
    partes = []
    for nombre, correccion in predicciones.items():
        d = df[columnas].reset_index(drop=True).copy()
        d['conjunto'] = conjunto
        d['modelo'] = nombre
        d['correccion_c'] = np.asarray(correccion)
        d['temperatura_predicha_c'] = d.temperature_forecast_c + d.correccion_c
        d['residual_c'] = d.temperature_observed_c - d.temperatura_predicha_c
        d['error_absoluto_c'] = d.residual_c.abs()
        d['error_cuadrado_c2'] = d.residual_c**2
        partes.append(d)
    return pd.concat(partes, ignore_index=True)

def agregar_metricas(detalle, dimensiones):
    claves = ['conjunto', 'modelo'] + dimensiones
    m = detalle.groupby(claves, dropna=False, observed=True).agg(
        n=('residual_c', 'size'), suma_error_absoluto_c=('error_absoluto_c', 'sum'),
        suma_error_cuadrado_c2=('error_cuadrado_c2', 'sum'), suma_residual_c=('residual_c', 'sum'),
        sin_lag=('lag_error_faltante_flag', 'sum')).reset_index()
    m['MAE_c'] = m.suma_error_absoluto_c/m.n
    m['RMSE_c'] = np.sqrt(m.suma_error_cuadrado_c2/m.n)
    m['sesgo_c'] = m.suma_residual_c/m.n
    m['cobertura_lag_pct'] = 100*(1-m.sin_lag/m.n)
    join = ['conjunto'] + dimensiones
    base = m.loc[m.modelo.eq('Original'), join+['MAE_c']].rename(columns={'MAE_c':'MAE_original_c'})
    m = m.merge(base, on=join, how='left', validate='many_to_one')
    m['mejora_MAE_pct'] = 100*(1-m.MAE_c/m.MAE_original_c.where(m.MAE_original_c.ne(0)))
    return m

detalle_val = detalle_predicciones(val, pred_val, 'validacion')
metricas_val = agregar_metricas(detalle_val, ['horizon_target_hours'])
ranking = metricas_val.groupby('modelo').MAE_c.mean().sort_values(kind='stable')
seleccionado = ranking.index[0]
display(metricas_val.round(4))
print('Configuración seleccionada exclusivamente en validación:', seleccionado)


## 5. Congelar la configuración y evaluar noviembre

Se vuelve a ajustar la configuración seleccionada con etiquetas históricas disponibles antes del corte final. Se carga test una sola vez, se valida, se construyen sus features y se predice sin reajustar nada. Los errores de noviembre aparecen como eventos únicamente cuando son anteriores a cada emisión. No se prueban nuevas configuraciones a partir del resultado de test.


In [ ]:
train_final = features_historicas.loc[features_historicas.error_disponible_utc.lt(CORTE_FINAL)].copy()
exigir(not train_final.empty, 'Entrenamiento final vacío')
modelo_final = None
if seleccionado != 'Original':
    modelo_final = clone(modelos[seleccionado])
    columnas_final = PREDICTORES if CONFIGURACIONES[seleccionado][1] else BASE
    modelo_final.fit(train_final[columnas_final], train_final.error_c)
# Se guarda también la selección si ganó el original, que no requiere estimador.
joblib.dump({'modelo': modelo_final, 'nombre': seleccionado, 'predictores':
             columnas_final if modelo_final is not None else [], 'corte_utc': str(CORTE_FINAL)},
            SALIDA / 'modelo_final.joblib')

test_crudo, calidad_test = cargar('test')
exigir(test_crudo.momento_emision_utc.ge(CORTE_FINAL).all(), 'Test contiene emisiones anteriores al corte final')
fuente_test = pd.concat([historico, test_crudo], ignore_index=True)
exigir(not fuente_test.duplicated(CLAVE).any(), 'Claves duplicadas entre histórico y test')
test = incorporar_lags(construir_features_meteorologicas(test_crudo), fuente_test)
pred_test = {'Original': np.zeros(len(test))}
if modelo_final is not None:
    pred_test[seleccionado] = modelo_final.predict(test[columnas_final])
    exigir(np.isfinite(pred_test[seleccionado]).all(), 'Predicciones test no finitas')
detalle_test = detalle_predicciones(test, pred_test, 'test_retrospectivo')
metricas_test = agregar_metricas(detalle_test, ['horizon_target_hours'])
display(metricas_test.round(4))
print('Test:', len(test), 'filas; fechas:', test.fecha.nunique(), '; modelo:', seleccionado)


## 6. Robustez temporal para la presentación

La dispersión diaria importa más que una única cifra global. El bootstrap pareado remuestrea **días completos** de test, mantiene ambos modelos sobre las mismas filas y recalcula el MAE ponderado por registros. El intervalo describe la variabilidad observada en estos pocos días; no garantiza generalización ni corrige selección previa. No se usan filas individuales como si fueran independientes. Se omite la comparación si ganó el original.


In [ ]:
metricas_diarias_test = agregar_metricas(detalle_test, ['fecha', 'horizon_target_hours'])
bootstrap = []
rng = np.random.default_rng(SEED)
if seleccionado != 'Original':
    for h in [12, 24]:
        t = metricas_diarias_test.loc[metricas_diarias_test.horizon_target_hours.eq(h)]
        o = t.loc[t.modelo.eq('Original')].set_index('fecha').sort_index()
        c = t.loc[t.modelo.eq(seleccionado)].set_index('fecha').reindex(o.index)
        indices = rng.integers(0, len(o), size=(2000, len(o)))
        suma_ventaja = (o.suma_error_absoluto_c-c.suma_error_absoluto_c).to_numpy()
        ventajas = suma_ventaja[indices].sum(axis=1)/o.n.to_numpy()[indices].sum(axis=1)
        bootstrap.append({'horizonte_h': h, 'dias': len(o), 'replicas': 2000,
                          'ventaja_MAE_c': suma_ventaja.sum()/o.n.sum(),
                          'p025_c': np.quantile(ventajas, .025), 'p975_c': np.quantile(ventajas, .975)})
robustez = pd.DataFrame(bootstrap, columns=['horizonte_h','dias','replicas','ventaja_MAE_c','p025_c','p975_c'])
display(robustez)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metricas_test.pivot(index='horizon_target_hours', columns='modelo', values='MAE_c').plot.bar(ax=axes[0])
axes[0].set(title='MAE en test retrospectivo', xlabel='Horizonte nominal (h)', ylabel='MAE (°C)')
if seleccionado != 'Original':
    t = metricas_diarias_test.loc[metricas_diarias_test.modelo.eq(seleccionado)]
    t.pivot(index='fecha', columns='horizon_target_hours', values='mejora_MAE_pct').plot(marker='o', ax=axes[1])
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set(title='Mejora diaria frente al original', ylabel='Reducción del MAE (%)', xlabel='Fecha objetivo UTC')
else:
    axes[1].text(.1,.5,'El original ganó en validación.\nNo se aplica corrección.', transform=axes[1].transAxes)
for ax in axes:
    ax.grid(axis='y', alpha=.25)
fig.tight_layout()
fig.savefig(SALIDA / 'resumen_test.png', dpi=160, bbox_inches='tight')
plt.show()


## 7. Contrato de datos del dashboard y conexión con 05/06

| Archivo | Uso |
|---|---|
| `metricas_globales.csv`, `metricas_horizonte.csv` | Tarjetas y comparación de modelos por conjunto |
| `metricas_diarias.csv` | Evolución temporal, caídas y mejoras |
| `metricas_estacion.csv` | Rendimiento por estación/horizonte; mostrar siempre `n` |
| `metricas_filtros.csv` | Filtros combinados por fecha, estación, horizonte y disponibilidad de lag |
| `predicciones_test.csv.gz` | Series, dispersión y detalle con coordenadas y trazabilidad |
| `dim_estacion.csv` | Identificadores y coordenadas medianas para el mapa; contar ubicaciones diferentes |
| `inventario.csv`, `calidad.csv`, `disponibilidad_historico.csv` | Cobertura, exclusiones y faltantes |
| `robustez_test.csv` | Intervalos bootstrap por día |
| `manifest.json`, `diccionario_datos.csv` | Protocolo, versiones, archivos y significado de columnas |
| `features_historicas_tp3.pkl.gz`, `variables_modelo_lags.json` | Intercambio histórico con notebooks 05/06 |

**Reglas de visualización:** no mezclar validación con test; las métricas de validación corresponden a modelos ajustados antes de validación y las de test al ajuste final. No promediar MAE o RMSE de grupos sin ponderar. Tras filtrar, usar `sum(suma_error_absoluto_c)/sum(n)`, `sqrt(sum(suma_error_cuadrado_c2)/sum(n))` y `sum(suma_residual_c)/sum(n)`. Calcular mejora contra el original sobre exactamente el mismo subconjunto. Una mejora negativa significa empeoramiento; si el MAE original es cero, la mejora porcentual no está definida. No sumar `n` entre modelos, porque evalúan las mismas filas. Todas las fechas son UTC.

El intercambio conserva los 23 predictores y agrega trazabilidad. **05/06 deben apuntar a esta ejecución y verificar el protocolo antes de consumirlo**; no se afirma compatibilidad de rutas o reglas sin revisar esos notebooks. Los pickle/joblib se cargan solo si son propios o de confianza. El test nunca se exporta dentro del histórico para CV.


In [ ]:
detalle = pd.concat([detalle_val, detalle_test], ignore_index=True)
tablas = {
    'metricas_globales.csv': agregar_metricas(detalle, []),
    'metricas_horizonte.csv': agregar_metricas(detalle, ['horizon_target_hours']),
    'metricas_diarias.csv': agregar_metricas(detalle, ['fecha','horizon_target_hours']),
    'metricas_estacion.csv': agregar_metricas(detalle, ['station_id','horizon_target_hours']),
    'metricas_filtros.csv': agregar_metricas(detalle, ['fecha','station_id','horizon_target_hours','lag_error_faltante_flag']),
    'inventario.csv': inventario,
    'calidad.csv': pd.concat([calidad_historico, calidad_test], ignore_index=True),
    'disponibilidad_historico.csv': disponibilidad.reset_index(),
    'particiones.csv': pd.concat([resumen_particiones, pd.DataFrame([
        {'conjunto':'train_final','filas':len(train_final)}, {'conjunto':'test','filas':len(test)}])], ignore_index=True),
    'robustez_test.csv': robustez
}
coordenadas = fuente_test[['station_id','latitude','longitude']].drop_duplicates()
dim = fuente_test.groupby('station_id').agg(latitude=('latitude','median'),longitude=('longitude','median'),
    state=('state','first')).reset_index()
dim = dim.merge(coordenadas.groupby('station_id').size().rename('ubicaciones_distintas').reset_index(), on='station_id')
tablas['dim_estacion.csv'] = dim
for nombre, tabla in tablas.items():
    tabla.to_csv(SALIDA/nombre, index=False, encoding='utf-8-sig')
detalle_test.to_csv(SALIDA/'predicciones_test.csv.gz', index=False, compression='gzip')
exigir(features_historicas.target_time_utc.lt(INICIO_TEST).all(), 'Test dentro del histórico exportado')
features_historicas.to_pickle(SALIDA/'features_historicas_tp3.pkl.gz', compression='gzip', protocol=4)
(SALIDA/'variables_modelo_lags.json').write_text(json.dumps(PREDICTORES, indent=2), encoding='utf-8')

significados = {
    'conjunto':'validacion o test_retrospectivo; no combinar métricas', 'modelo':'Nombre de configuración',
    'n':'Cantidad de predicciones del grupo para un modelo', 'MAE_c':'Error absoluto medio, °C',
    'RMSE_c':'Raíz del error cuadrático medio, °C', 'sesgo_c':'Media de observado menos predicho, °C',
    'MAE_original_c':'MAE del original en el mismo grupo', 'mejora_MAE_pct':'100*(1-MAE/MAE_original)',
    'suma_error_absoluto_c':'Suma de errores absolutos; numerador aditivo del MAE',
    'suma_error_cuadrado_c2':'Suma de cuadrados; numerador aditivo del MSE',
    'suma_residual_c':'Suma de observado menos predicho; numerador aditivo del sesgo',
    'sin_lag':'Cantidad de filas sin antecedente válido', 'cobertura_lag_pct':'Porcentaje de filas con lag válido',
    'station_id':'Identificador de estación; conservar como texto', 'state':'Estado informado en los CSV',
    'latitude':'Latitud en grados', 'longitude':'Longitud en grados',
    'fecha':'Fecha objetivo UTC, YYYY-MM-DD', 'horizon_target_hours':'Horizonte nominal respecto al snapshot, horas',
    'target_time_utc':'Fecha/hora objetivo UTC', 'momento_emision_utc':'Máximo de generación y captura, UTC',
    'temperature_forecast_c':'Pronóstico original, °C', 'temperature_observed_c':'Observación, °C',
    'correccion_c':'Error estimado sumado al pronóstico original, °C',
    'temperatura_predicha_c':'Pronóstico original más corrección, °C', 'residual_c':'Observado menos predicho, °C',
    'error_absoluto_c':'Valor absoluto del residual, °C', 'error_cuadrado_c2':'Cuadrado del residual, °C²',
    'lag_error_faltante_flag':'1 sin antecedente válido; 0 con antecedente',
    'lag_error_edad_h':'Horas desde objetivo del antecedente a emisión actual',
    'lag_disponible_utc':'Disponibilidad estimada del antecedente utilizado, UTC',
    'lag_objetivo_utc':'Objetivo del antecedente utilizado, UTC',
    'lag_origen':'historico o test; vacío si falta lag', 'archivo_origen':'CSV fuente de la fila'
}
columnas_dashboard = set(detalle_test.columns) | set(tablas['metricas_filtros.csv'].columns)
exigir(columnas_dashboard <= set(significados), 'Falta documentar una columna del dashboard')
pd.DataFrame([{'columna':c,'descripcion':significados[c]} for c in sorted(columnas_dashboard)]).to_csv(
    SALIDA/'diccionario_datos.csv', index=False, encoding='utf-8-sig')

def sha256_archivo(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for bloque in iter(lambda: f.read(1024*1024), b''):
            h.update(bloque)
    return h.hexdigest()

protocolo = {'version': VERSION, 'lags': 'secuencial_eventos_disponibles',
            'margen_observacion_h': MARGEN_H, 'edad_maxima_lag_h': EDAD_MAX_H,
            'coincidencias_exactas': False, 'empate_eventos':'objetivo_mas_reciente',
            'sin_busqueda_alternativa_si_ultimo_evento_invalido': True,
            'corte_validacion_objetivo_utc': str(CORTE_VAL), 'primera_emision_validacion_utc': str(inicio_val),
            'corte_ajuste_final_utc': str(CORTE_FINAL), 'inicio_test_objetivo_utc':str(INICIO_TEST),
            'excluidos':sorted(str(d.date()) for d in excluidas), 'seed':SEED,
            'seleccion':'menor media de MAE de 12 y 24 h en validacion',
            'interpretacion':'retrospectivo; test previamente explorado; seleccion solapa emisiones iniciales',
            'modelo_seleccionado': seleccionado}
manifest = {'estado':'completo', 'id_ejecucion':id_ejecucion, 'protocolo':protocolo,
            'versiones':{'python':platform.python_version(),'pandas':pd.__version__,
                         'numpy':np.__version__,'sklearn':sklearn.__version__},
            'filas':{'historico':len(historico),'train':len(train),'validacion':len(val),
                     'train_final':len(train_final),'test':len(test)},
            'parametros_modelos': {n:m.named_steps['modelo'].get_params() for n,m in modelos.items()},
            'fuentes':inventario.to_dict(orient='records'),
            'archivos':[{'nombre':p.name,'bytes':p.stat().st_size,'sha256':sha256_archivo(p)}
                        for p in sorted(SALIDA.iterdir()) if p.is_file()]}
# Verificación de estabilidad básica: no se modificaron fuentes durante la ejecución.
for f in inventario.itertuples():
    p = DATASET/f.archivo
    exigir(p.stat().st_size == f.bytes and p.stat().st_mtime_ns == f.mtime_ns, 'CSV modificado durante la ejecución')
(SALIDA/'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
print('Ejecución completa:', SALIDA)


## 8. Lectura final generada a partir de esta ejecución

La conclusión siguiente se calcula, no se copia de versiones anteriores. El modelo puede mejorar o empeorar. Para la exposición, acompañar el resultado global con los días que empeoraron, cobertura de lags y cantidad de registros. Un test de ocho días y una disponibilidad estimada no permiten afirmar funcionamiento en producción ni generalización a otras estaciones o estaciones del año.

Se conservaron la lógica de purga, los ciclos temporales, los modelos base y la comparación con/sin lags. La importancia por permutación del original no se usa aquí para afirmar qué variable es más influyente: exigiría recalcularla en una etapa específica de interpretación. Tampoco se inventan resultados de CV de 06. Los archivos de esta ejecución permiten construir el dashboard sin entrenar modelos desde su interfaz.


In [ ]:
global_test = tablas['metricas_globales.csv'].query("conjunto == 'test_retrospectivo'").set_index('modelo')
o = global_test.loc['Original']
c = global_test.loc[seleccionado]
print(f'Modelo seleccionado: {seleccionado}.')
print(f'Test retrospectivo: {len(test):,} observaciones, {test.fecha.nunique()} días, {test.station_id.nunique()} estaciones.')
print(f'MAE original: {o.MAE_c:.4f} °C; MAE seleccionado: {c.MAE_c:.4f} °C.')
print(f'Cambio porcentual favorable del MAE: {c.mejora_MAE_pct:.2f} %. Valores negativos indican empeoramiento.')
print(f'RMSE seleccionado: {c.RMSE_c:.4f} °C; sesgo observado menos predicho: {c.sesgo_c:+.4f} °C.')
dias = metricas_diarias_test.loc[metricas_diarias_test.modelo.eq(seleccionado)]
print(f'Días × horizonte con mejora: {(dias.mejora_MAE_pct > 0).sum()} de {len(dias)}; '
      f'con empeoramiento: {(dias.mejora_MAE_pct < 0).sum()}.')
print('No es una prueba prospectiva inédita. No se retocaron parámetros utilizando esta ejecución de test.')
